In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
import pickle

# Load dataset
df = pd.read_csv("yield_df.csv")

# Remove unwanted column
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

print("Columns:", df.columns)

# Input features
X = df[[
    "Area",
    "Item",
    "Year",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes",
    "avg_temp"
]].copy()

# Output
y = df["hg/ha_yield"]

# Encode text columns
le_area = LabelEncoder()
le_item = LabelEncoder()

X.loc[:, "Area"] = le_area.fit_transform(X["Area"])
X.loc[:, "Item"] = le_item.fit_transform(X["Item"])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Train model
model = RandomForestRegressor(n_estimators=200, random_state=42)
model.fit(X_train, y_train)

# Save model & encoders
pickle.dump(model, open("crop_model.pkl", "wb"))
pickle.dump(le_area, open("le_area.pkl", "wb"))
pickle.dump(le_item, open("le_item.pkl", "wb"))

print("Training Done!")
print("Model expects features:", model.n_features_in_)


Columns: Index(['Area', 'Item', 'Year', 'hg/ha_yield', 'average_rain_fall_mm_per_year',
       'pesticides_tonnes', 'avg_temp'],
      dtype='object')
Training Done!
Model expects features: 6


In [1]:
import pandas as pd
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pickle

# Load dataset
df = pd.read_csv("yield_df.csv")

# Remove unwanted column
if "Unnamed: 0" in df.columns:
    df.drop(columns=["Unnamed: 0"], inplace=True)

# Input features
X = df[[
    "Area",
    "Item",
    "Year",
    "average_rain_fall_mm_per_year",
    "pesticides_tonnes",
    "avg_temp"
]].copy()

# Target variable
y = df["hg/ha_yield"]

# Encode categorical features
le_area = LabelEncoder()
le_item = LabelEncoder()
X["Area"] = le_area.fit_transform(X["Area"])
X["Item"] = le_item.fit_transform(X["Item"])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Load trained model (or train again)
model = pickle.load(open("crop_model.pkl", "rb"))

# Make predictions
y_pred = model.predict(X_test)

# Evaluation metrics
mae = mean_absolute_error(y_test, y_pred)
mse = mean_squared_error(y_test, y_pred)
rmse = np.sqrt(mse)
r2 = r2_score(y_test, y_pred)

print("Model Evaluation Metrics:")
print(f"Mean Absolute Error (MAE): {mae:.2f} hg/ha")
print(f"Mean Squared Error (MSE): {mse:.2f} hg²/ha²")
print(f"Root Mean Squared Error (RMSE): {rmse:.2f} hg/ha")
print(f"R² Score: {r2:.2f}")

Model Evaluation Metrics:
Mean Absolute Error (MAE): 3741.53 hg/ha
Mean Squared Error (MSE): 103842314.53 hg²/ha²
Root Mean Squared Error (RMSE): 10190.30 hg/ha
R² Score: 0.99
